# Tiny Critic: Compile Evidence into a Portable Judgement

## What this demonstrates

This notebook demonstrates a bounded ZeroModel Critic mechanism:

```text
identified evidence
        ↓
declared feature representation
        ↓
tiny specialised critic readout
        ↓
score + explanation
        ↓
rank / triage
        ↓
VPM
        ↓
portable runtime
        ↓
receipt + replay
```

The fixture demonstrates the Critic mechanism. It does not establish that these synthetic feature names correspond to real reasoning quality, writing quality, correctness, or truth.

## Why it matters

Large systems can produce expensive evidence or metrics. Often the operational question is narrow: which candidate should be reviewed first, which run looks risky, or which evidence resembles known successful cases?

ZeroModel's design is to prepare rich identified evidence once, then attach small specialised consumers.

```text
                 IDENTIFIED REPRESENTATION
                          │
          ┌───────────────┼───────────────┐
          │               │               │
        Search           Critic          Policy
          │               │               │
   relation readout   judgement readout  action rule
          │               │               │
    "related how?"    "looks good/risky?" "what do?"
```

Search asks a relation-specific question. Critic asks a judgement-specific question. Policy asks an action question. They are not the same algorithm; the shared architectural move is prepared evidence plus small declared consumers.

## Source and package mapping

- Source example: `examples/tiny_critic.py`
- Packages: `zeromodel.core`, `zeromodel.artifacts`, `zeromodel.critic`
- No network, external model, sklearn, PyTorch, or random fixture generation is used.

In [ ]:
from pathlib import Path
import sys

import json
import math
import numpy as np

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'examples' / 'tiny_critic.py').is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError('could not locate repository root')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.tiny_critic import ( # NOQA: E402
    EVALUATION_RECORDS,
    TRAINING_RECORDS,
    baseline_metrics,
    independent_portable_score,
    run_tiny_critic_fixture,
    store_fixture_batch,
)
from zeromodel.critic import ( # NOQA: E402
    CriticFeatureBatchDTO,
    CriticFeatureDTO,
    CriticFeatureSchemaMismatchError,
    CriticFeatureSpecDTO,
    CriticFitSpecDTO,
    CriticScoreRequestDTO,
    build_critic_score_vpm,
    compile_critic_readout,
    evaluate_binary_critic,
    evaluate_promotion,
    load_critic_evaluation_result_aggregate,
    load_critic_readout_aggregate,
    rank_by_critic,
    replay_critic_score,
    score_critic,
    score_portable,
)
from zeromodel.critic.persistence import ( # NOQA: E402
    load_critic_feature_batch,
    load_critic_feature_batch_aggregate,
    load_critic_label_batch,
    load_critic_label_batch_aggregate,
    load_critic_score_result,
    store_critic_feature_batch,
    store_critic_feature_spec,
)
from zeromodel.critic.promotion import CriticPromotionPolicyDTO # NOQA: E402
from zeromodel.critic.scoring import compiled_from_aggregate # NOQA: E402

## Application

The application in this notebook is a deterministic synthetic capability fixture. It uses four declared numeric signals:

| Feature | Direction | Meaning in this fixture |
|---|---:|---|
| stability | +1 | higher is associated with success |
| coverage | +1 | higher is associated with success |
| uncertainty | -1 | lower is associated with success |
| consistency | +1 | higher is associated with success |

These names are not truths. They are just declared numeric signals for this fixture. The critic package is metric-agnostic.

In [ ]:
run = run_tiny_critic_fixture()
feature_spec = run.feature_spec
contract = run.contract

feature_rows = [
    {
        "feature": feature.feature_id,
        "directionality": feature.directionality,
        "required": feature.required,
        "missing_policy": feature.missing_policy,
        "description": feature.description,
    }
    for feature in feature_spec.features
]
feature_rows

## Declare what the critic means

The critic answers one declared target: `synthetic-success`. Its score semantics are similarity to successful rows in this deterministic fixture. The contract explicitly prohibits semantic truth, universal quality, real reasoning correctness, and text quality claims.

In [ ]:
contract.to_dict()

## Build identified training evidence

The fixture has separate training and held-out batches. Rows are stored as content-addressed item artifacts; the batches own the row binding.

In [ ]:
print("training rows:", len(TRAINING_RECORDS))
print("held-out rows:", len(EVALUATION_RECORDS))
print("feature_spec_id:", run.feature_spec.feature_spec_id)
print("critic_contract_id:", run.contract.critic_contract_id)
print("training feature batch:", run.training.feature_ref.artifact_id)
print("training label batch:", run.training.label_ref.artifact_id)
print("held-out feature batch:", run.evaluation.feature_ref.artifact_id)
print("held-out label batch:", run.evaluation.label_ref.artifact_id)
assert run.training.feature_ref.artifact_id != run.evaluation.feature_ref.artifact_id
assert {ref.artifact_id for ref in run.training.item_refs}.isdisjoint({ref.artifact_id for ref in run.evaluation.item_refs})
TRAINING_RECORDS[:5]

## Compile the tiny critic

The v1 readout is deliberately small: directionality, standardisation, a linear logit, and a sigmoid score. Coefficients are contributions to this critic's fitted boundary, not causal explanations.

In [ ]:
aggregate = load_critic_readout_aggregate(run.readout_ref, run.store)
runtime = compiled_from_aggregate(aggregate)
coefficient_rows = [
    {"feature": fid, "directionality": direction, "coefficient": float(coef)}
    for fid, direction, coef in zip(runtime.feature_ids, runtime.directionality, runtime.coefficients)
]
print("readout artifact:", run.readout_ref.artifact_id)
print("fit spec:", run.fit_spec.to_dict())
print("feature count:", len(runtime.feature_ids))
print("coefficient count:", len(runtime.coefficients))
coefficient_rows

## Inspect the actual arithmetic

For one held-out row, feature contributions are exactly:

```text
standardized_feature_i * coefficient_i
```

The sum of contributions plus the intercept equals the logit. Applying the sigmoid to the logit yields the critic score when no calibration artifact is attached.

In [ ]:
score_result = load_critic_score_result(run.store, run.score_result_ref)
row = run.evaluation.values[0]
item = score_result.items[0]
contributions = runtime.contributions_one(row, feature_spec_id=run.feature_spec.feature_spec_id)
contribution_rows = [c.to_dict() for c in contributions]
logit_from_parts = sum(c.contribution for c in contributions) + runtime.intercept
score_from_logit = 1.0 / (1.0 + math.exp(-logit_from_parts))
assert logit_from_parts == np.float64(item.logit).item() or abs(logit_from_parts - item.logit) < 1e-10
assert abs(score_from_logit - item.score) < 1e-10
print("artifact:", item.artifact_ref.artifact_id)
print("logit:", item.logit)
print("score:", item.score)
contribution_rows

## Held-out evidence

Scoring and evaluation use the held-out batch. The held-out rows never participate in fitting.

In [ ]:
assert score_result.feature_batch_ref.artifact_id == run.evaluation.feature_ref.artifact_id
assert score_result.feature_batch_ref.artifact_id != run.training.feature_ref.artifact_id
print("training batch ID:", run.training.feature_ref.artifact_id)
print("held-out batch ID:", run.evaluation.feature_ref.artifact_id)
print("row sets disjoint:", {ref.artifact_id for ref in run.training.item_refs}.isdisjoint({ref.artifact_id for ref in run.evaluation.item_refs}))

## Score the held-out set

The score is not truth. The verdict is a declared downstream threshold policy over the score.

In [ ]:
heldout_table = []
for record, scored in zip(EVALUATION_RECORDS, score_result.items):
    heldout_table.append({
        "candidate": record["id"],
        "label": "successful" if record["label"] else "failed",
        "score": round(scored.score, 6),
        "verdict": scored.verdict,
        "margin": round(scored.decision_margin, 6),
    })
heldout_table

## Ordinary baselines

A skeptical reader should ask why not use one scalar or a hand aggregate. These baselines are fixed by the fixture design: coverage-only and the mean of direction-corrected features.

In [ ]:
baselines = baseline_metrics(run.evaluation.labels, run.evaluation.values)
{
    "random_expected_positive_rate": baselines["random_expected_positive_rate"],
    "coverage_only": baselines["coverage_only"],
    "direction_corrected_mean": baselines["direction_corrected_mean"],
}

## Evaluation metrics

The uncalibrated sigmoid output is reported as a critic score, not as a calibrated probability.

In [ ]:
scores = np.asarray([item.score for item in score_result.items], dtype=np.float64)
metrics = evaluate_binary_critic(run.evaluation.labels, scores, bin_count=5)
assert metrics == dict(run.metrics)
metrics

## Persisted evaluation evidence

The evaluation measurement is not only notebook output. It is bound to an identified held-out evaluation set and an identified evaluation result artifact.

In [ ]:
evaluation_aggregate = load_critic_evaluation_result_aggregate(run.evaluation_result_ref, run.store)
print("critic readout ID:", run.readout_ref.artifact_id)
print("evaluation set ID:", run.evaluation_set_ref.artifact_id)
print("evaluation result ID:", run.evaluation_result_ref.artifact_id)
assert evaluation_aggregate.evaluation_result.readout_ref.artifact_id == run.readout_ref.artifact_id
assert evaluation_aggregate.evaluation_set.evaluation_set.feature_batch_ref.artifact_id == run.evaluation.feature_ref.artifact_id
assert evaluation_aggregate.evaluation_set.evaluation_set.label_batch_ref.artifact_id == run.evaluation.label_ref.artifact_id
evaluation_aggregate.evaluation_result.to_dict()

## Ranking and triage

Tiny Critic is a triage signal. It does not replace expensive verification.

In [ ]:
ranked_ids = rank_by_critic(score_result)
rank_rows = []
for rank, artifact_id in enumerate(ranked_ids, start=1):
    index = [item.artifact_ref.artifact_id for item in score_result.items].index(artifact_id)
    rank_rows.append({"rank": rank, "candidate": EVALUATION_RECORDS[index]["id"], "score": round(score_result.items[index].score, 6), "label": EVALUATION_RECORDS[index]["label"]})
assert ranked_ids == tuple(sorted(ranked_ids, key=lambda aid: (-next(item.score for item in score_result.items if item.artifact_ref.artifact_id == aid), aid)))
rank_rows

In [ ]:
budget_rows = tuple(run.budget_rows)
budget_rows

## VPM: turn judgement into an inspectable sign

The critic produces the score. The VPM organises the result into an identified, inspectable operational view. It does not create the judgement.

In [ ]:
vpm = build_critic_score_vpm(result=score_result)
assert vpm.artifact_id == run.vpm_artifact_id
assert vpm.provenance["readout_ref"] == run.readout_ref.artifact_id
assert vpm.provenance["feature_batch_ref"] == run.evaluation.feature_ref.artifact_id
top_cell = vpm.cell(0, 0)
top_index = [item.artifact_ref.artifact_id for item in score_result.items].index(top_cell.row_id)
top_item = score_result.items[top_index]
all_contribs = runtime.contributions_one(run.evaluation.values[top_index], feature_spec_id=run.feature_spec.feature_spec_id)
pos = max(all_contribs, key=lambda c: (c.contribution, c.feature_id))
neg = min(all_contribs, key=lambda c: (c.contribution, c.feature_id))
{
    "vpm_artifact_id": vpm.artifact_id,
    "top_left_candidate": EVALUATION_RECORDS[top_index]["id"],
    "source_artifact_ref": top_item.artifact_ref.artifact_id,
    "critic_score": top_item.score,
    "verdict": top_item.verdict,
    "dominant_positive_contribution": pos.to_dict(),
    "dominant_negative_contribution": neg.to_dict(),
}

## Review view

For review priority, use an explicit rule: distance to the nearest decision threshold. Smaller distance means more borderline.

In [ ]:
thresholds = [0.40, 0.60]
review_rows = []
for record, scored in zip(EVALUATION_RECORDS, score_result.items):
    distance = min(abs(scored.score - threshold) for threshold in thresholds)
    review_rows.append({"candidate": record["id"], "score": round(scored.score, 6), "nearest_threshold_distance": round(distance, 6), "verdict": scored.verdict})
sorted(review_rows, key=lambda row: (row["nearest_threshold_distance"], row["candidate"]))

## Feature-schema safety

ZeroModel refuses to silently reinterpret a critic against a different evidence schema.

In [ ]:
original_batch = load_critic_feature_batch(run.store, run.evaluation.feature_ref)
changed_spec = CriticFeatureSpecDTO(
    features=(
        CriticFeatureDTO("coverage", "changed order"),
        CriticFeatureDTO("stability", "changed order"),
        CriticFeatureDTO("uncertainty", "changed order", directionality=-1),
        CriticFeatureDTO("consistency", "changed order"),
    )
)
changed_spec_ref = store_critic_feature_spec(run.store, changed_spec)
incompatible_batch = CriticFeatureBatchDTO(
    feature_spec_ref=changed_spec_ref,
    values_blob_ref=original_batch.values_blob_ref,
    item_refs=original_batch.item_refs,
    values_shape=original_batch.values_shape,
    values_dtype=original_batch.values_dtype,
)
incompatible_ref = store_critic_feature_batch(run.store, incompatible_batch)
try:
    score_critic(store=run.store, request=CriticScoreRequestDTO(readout_ref=run.readout_ref, feature_batch_ref=incompatible_ref))
except CriticFeatureSchemaMismatchError as exc:
    print(type(exc).__name__, str(exc))
else:
    raise AssertionError("schema mismatch was not rejected")

## Portable critic

The executable payload is canonical JSON containing only arithmetic inputs: feature IDs, directionality, centre, scale, coefficients, intercept, contract identities, and optional calibration.

In [ ]:
payload = run.portable_payload
payload_bytes = len(payload.encode("utf-8"))
limit = run.fit_spec.portable_payload_limit_bytes
payload_summary = json.loads(payload)
assert payload_bytes < limit
{
    "payload_bytes": payload_bytes,
    "declared_limit_bytes": limit,
    "percent_of_limit": round(100.0 * payload_bytes / limit, 3),
    "feature_ids": payload_summary["feature_ids"],
    "has_calibration": payload_summary["calibration"] is not None,
    "keys": sorted(payload_summary.keys()),
}

## Independent portable arithmetic

This cell compares the normal runtime, package portable runtime, and a notebook-local arithmetic reference.

In [ ]:
for index in (0, 4, 7):
    normal = score_result.items[index]
    portable = score_portable(payload, run.evaluation.values[index].tolist())
    reference = independent_portable_score(payload, run.evaluation.values[index])
    assert abs(normal.score - portable["score"]) < 1e-10
    assert abs(normal.score - reference["score"]) < 1e-10
print("portable arithmetic agrees for rows 0, 4, and 7")

## Identity mutation

Flip one training label, rebuild the critic, and evaluate the candidate on the same frozen held-out evaluation evidence.

In [ ]:
mutated_records = list(TRAINING_RECORDS)
mutated_record = dict(mutated_records[0])
mutated_record["label"] = 0
mutated_records[0] = mutated_record
training_feature_batch = load_critic_feature_batch(run.store, run.training.feature_ref)
training_label_batch = load_critic_label_batch(run.store, run.training.label_ref)
mutated_training = store_fixture_batch(
    run.store,
    split="training-mutated-notebook",
    records=tuple(mutated_records),
    feature_spec_ref=training_feature_batch.feature_spec_ref,
    contract_ref=training_label_batch.critic_contract_ref,
)
_, mutated_readout_ref = compile_critic_readout(
    store=run.store,
    features=load_critic_feature_batch_aggregate(mutated_training.feature_ref, run.store),
    labels=load_critic_label_batch_aggregate(mutated_training.label_ref, run.store),
    fit_spec=CriticFitSpecDTO(l2_penalty=0.1, max_iterations=80),
)
assert mutated_training.label_ref.artifact_id != run.training.label_ref.artifact_id
assert mutated_readout_ref.artifact_id != run.readout_ref.artifact_id
{
    "original_label_batch": run.training.label_ref.artifact_id,
    "mutated_label_batch": mutated_training.label_ref.artifact_id,
    "original_readout": run.readout_ref.artifact_id,
    "mutated_readout": mutated_readout_ref.artifact_id,
}

## Candidate critic evaluation

Current and candidate critics are scored against the exact same held-out feature batch.

In [ ]:
mutated_request = CriticScoreRequestDTO(
    readout_ref=mutated_readout_ref,
    feature_batch_ref=run.evaluation.feature_ref,
)
mutated_result, mutated_result_ref, _ = score_critic(store=run.store, request=mutated_request)
assert mutated_result_ref is not None
mutated_scores = np.asarray([item.score for item in mutated_result.items], dtype=np.float64)
mutated_metrics = evaluate_binary_critic(run.evaluation.labels, mutated_scores, bin_count=5)
comparison = [
    {"metric": key, "current": run.metrics[key], "candidate": mutated_metrics[key]}
    for key in ("auroc", "accuracy", "brier", "ece")
]
assert mutated_result.feature_batch_ref.artifact_id == run.evaluation.feature_ref.artifact_id
comparison

## Promotion policy

Promotion is a recommendation over frozen evaluation evidence. It is not deployment authority.

In [ ]:
decision = evaluate_promotion(
    current_metrics={key: float(value) for key, value in run.metrics.items() if isinstance(value, (float, int))},
    candidate_metrics={key: float(value) for key, value in mutated_metrics.items() if isinstance(value, (float, int))},
    policy=CriticPromotionPolicyDTO(min_candidate_auroc=0.70, max_candidate_ece=0.30, min_auroc_gain=0.0),
)
assert isinstance(decision.recommended, bool)
{"recommended": decision.recommended, "reasons": decision.reasons}

## Receipt and exact replay

The receipt reloads the request, readout, and held-out feature batch, recomputes scoring, and checks the exact result identity.

In [ ]:
replayed = replay_critic_score(store=run.store, receipt_ref=run.receipt_ref)
assert replayed.result_id == score_result.result_id
{"receipt": run.receipt_ref.artifact_id, "replay_result_id": replayed.result_id}

## Money shot

Large prepared evidence collapses into a tiny identified operational judgement.

In [ ]:
top_id = rank_by_critic(score_result)[0]
top_index = [item.artifact_ref.artifact_id for item in score_result.items].index(top_id)
top_item = score_result.items[top_index]
all_contribs = runtime.contributions_one(run.evaluation.values[top_index], feature_spec_id=run.feature_spec.feature_spec_id)
pos = max(all_contribs, key=lambda c: (c.contribution, c.feature_id))
neg = min(all_contribs, key=lambda c: (c.contribution, c.feature_id))
{
    "critic": run.readout_ref.artifact_id,
    "target": run.contract.target_id,
    "held_out_evaluation": run.evaluation_set_ref.artifact_id,
    "top_candidate": EVALUATION_RECORDS[top_index]["id"],
    "critic_score": top_item.score,
    "verdict": top_item.verdict,
    "largest_positive_contribution": f"{pos.feature_id} {pos.contribution:+.6f}",
    "largest_negative_contribution": f"{neg.feature_id} {neg.contribution:+.6f}",
    "portable_payload": f"{payload_bytes} / {limit} bytes",
    "vpm": run.vpm_artifact_id,
    "replay": "IDENTICAL",
}

## Boundaries and limitations

| Capability | Demonstrated? | Boundary |
|---|---:|---|
| Declared numeric feature schema | Yes | Synthetic fixture only |
| Deterministic tiny logistic readout | Yes | Binary linear v1 |
| Separate held-out scoring | Yes | Deterministic synthetic fixture |
| Feature-contribution closure | Yes | Linear-model contribution, not causal explanation |
| Feature schema mismatch rejection | Yes | Declared critic schema |
| Ranking and triage | Yes | Fixture labels only |
| VPM score view | Yes | Organises judgement; does not create it |
| Portable arithmetic-only inference | Yes | Current portable v1 contract |
| Sub-50KiB payload | Measured | This critic payload only |
| Mutation changes critic identity | Yes | Controlled fixture mutation |
| Candidate promotion decision | Yes | Explicit fixture policy |
| Receipt replay | Yes | Same identified artifacts |
| Text quality detection | No | Future Writer experiment |
| Reasoning correctness detection | No | Historical research requires revalidation |
| Hallucination detection | No | Separate future critic |
| Universal quality score | No | Explicitly prohibited |

The fixture demonstrates the Critic mechanism. It does not establish that these synthetic feature names correspond to real reasoning quality, writing quality, correctness, or truth.

A Tiny Critic is not powerful because logistic regression is intelligent. It is useful because upstream systems have transformed a difficult problem into a small, stable, identified evidence space in which one narrow judgement can be represented by a tiny readout.

## Reproduction record

Run from the repository root:

```bash
python examples/tiny_critic.py
python scripts/build_demos.py validate
python scripts/build_demos.py execute --profile fast
```

This notebook is deterministic, offline, and requires no secrets, external models, or network access.